In [ ]:
  import time
import os
import time
import xlwt
from xlwt.Workbook import *

from datetime import date, timedelta, datetime
import openpyxl as opxl
import warnings
import numpy as np
import pandas as pd
from pandas import ExcelWriter
import xlsxwriter

"""
En realidad no creo usar todas las librerias, es como el 'paquete' usual que uso
"""

warnings.filterwarnings('ignore')

In [ ]:
direccion = 'D:/Archivos/DS4A/DS4A/ Colombia - Cohort 6/Project/'

datos1 = direccion + 'DB_inspeccion2020.xlsx'   # para el 2020
datos2 = direccion + 'DB_inspeccion2021.xlsx'   # para el 2021
columns = direccion + 'columnas_insp.xlsx'      # datos de las columnas
resultado1 = direccion + 'resultado01.xlsx'     # donde se imprimen los datos

In [ ]:
# df = pd.read_excel(datos1, index_col= False)
datos_subitems = pd.read_excel(columns, index_col= False, sheet_name= 'Subitems')       # como se llaman los subitems y como se renombraran
datos_columnas = pd.read_excel(columns, index_col= False, sheet_name= 'Tipo1')
    
dict_columnas = {}

for cols in datos_columnas['Nombre'].tolist():
    dict_columnas[cols] = datos_columnas.loc[datos_columnas['Nombre'] == cols, 'Nuevo'].tolist()[0]

prueba = pd.ExcelFile(datos2)
nombres = prueba.sheet_names

In [ ]:
lista_subitems = datos_subitems['Valores'].tolist()     # todos los posibles nombres de los subitems
todos_parametros = []

columnas_totales = ['Year', 'Establecimiento', 'ID_Control', 'Fecha', 'Razon_social', 'Nombre_comercial', 'N_documento', 'Departamento', 'Municipio', 'Zona', 'Barrio', 'Cumplimiento']
lista_calificaciones = datos_subitems['Nombre'].drop_duplicates().tolist()
for lcal in lista_calificaciones:
    cat = datos_subitems.loc[datos_subitems['Nombre'] == lcal, 'Categoria'].tolist()[0]
    columnas_totales.append(cat + ':' + lcal)
    columnas_totales.append(cat + ':' + lcal + '_VALORES')
columnas_totales.append('Calificacion_Total')
database = pd.DataFrame(columns= columnas_totales)
_n_columnas = len(database.columns.tolist())

for nombre_pestana in nombres:

    # Diccionarios para los numeros y localizacion de pestans y columnas
    parametros_pestana = {}
    loc_columnas = {}
    loc_calificaciones = {}

    # se crea un dataframe para cada pestana
    print(nombre_pestana)
    df = pd.read_excel(datos2, index_col= False, sheet_name= nombre_pestana)
    df = df.rename(columns= dict_columnas)
    df.fillna(value="", inplace= True)
    # print(df.columns)
    df2 = df.copy(deep= True)
    df2.drop(df.index[[0, 1]], inplace= True)       # aqui se detectaran los valores quitando las filas 1 y 2
    print(df.columns.tolist())

    # agrega en numero y el nombre de la columna
    contador_col = 0
    for i2 in df.columns:
        loc_columnas[contador_col] = i2
        contador_col += 1
    

    # sub dataframe para ver los nombres de los sub-items a calificar
    _df1 = df.loc[0, :]

    for i in range(_df1.shape[0]):  # agregando los nombres de subitems
        if str(_df1[i]) in lista_subitems:
            parametros_pestana[str(_df1[i])] = i

    for i3 in range(df.shape[1]):
        val_calif = df.loc[1, loc_columnas[i3]]
        loc_calificaciones[i3] = val_calif

    # para tener el valor de la llave para un diccionario
    def get_key(diccionario, v):
        for key, valores in diccionario.items():
            if v == valores:
                return key

    # solo interesan los parametros, entonces las columnas interes son las que estan igual o despues de edificaciones, o antes de cumplimiento
    num_cumplimiento = get_key(loc_columnas, 'Cumplimiento') - 1    # Hasta antes de la columna cumplimiento se tienen los items de calificacion
    num_param1 = get_key(loc_columnas, 'EDIFICACIONES')         # Desde la columna EDIFICACIONES inician los items de calificacion

    columnas_identificacion = []
    for n_def in range(num_param1):
        columnas_identificacion.append(loc_columnas[n_def])

    nums_columnas = list(parametros_pestana.values())
    nums_columnas_parm = []
    for k in nums_columnas:
        if k >= num_param1 and k < num_cumplimiento:
            nums_columnas_parm.append(k)
    del(nums_columnas)

    
    # lista de los numeros de id de acta de control
    lista_calif_establ = pd.DataFrame(columns= database.columns.tolist())
    # ['Year', 'Establecimiento', 'ID_Control', 'Fecha', 'Razon_social', 'Nombre_comercial', 'N_documento', 'Departamento', 'Municipio', 'Zona', 'Barrio', 'Cumplimiento']
    for k_id in df2['ID_Control'].tolist():
        fecha1_ = 2021
        estab_ = nombre_pestana
        id_ = k_id
        fecha2_ = df.loc[df['ID_Control'] == k_id, 'Fecha_visita'].astype(str).tolist()[0] if 'Fecha_visita' in df.columns.tolist() else df.loc[df['ID_Control'] == k_id, 'Fecha_ingreso'].astype(str).tolist()[0]
        razon_ = df.loc[df['ID_Control'] == k_id, 'Razon_social'].tolist()[0] if 'Razon_social' in df.columns.tolist() else 'NA'
        comercial_ = df.loc[df['ID_Control'] == k_id, 'Nombre_comercial'].tolist()[0] if 'Nombre_comercial' in df.columns.tolist() else 'NA'
        documento_ = df.loc[df['ID_Control'] == k_id, 'N_documento'].tolist()[0] if 'N_documento' in df.columns.tolist() else 'NA'
        departamento_ = df.loc[df['ID_Control'] == k_id, 'Departamento'].tolist()[0] if 'Departamento' in df.columns.tolist() else 'NA'
        municipio_ = df.loc[df['ID_Control'] == k_id, 'Municipio'].tolist()[0] if 'Municipio' in df.columns.tolist() else 'NA'
        zona_ = df.loc[df['ID_Control'] == k_id, 'Zona'].tolist()[0] if 'Zona' in df.columns.tolist() else 'NA'
        barrio_ = df.loc[df['ID_Control'] == k_id, 'Barrio'].tolist()[0] if 'Barrio' in df.columns.tolist() else 'NA'
        cumplimiento_ = df.loc[df['ID_Control'] == k_id, 'Cumplimiento'].tolist()[0] if 'Cumplimiento' in df.columns.tolist() else 'NA'
        new_fila = [fecha1_, estab_, id_,, fecha2_, razon_, comercial_, documento_, departamento_, municipio_, zona_, barrio_, cumplimiento_]
        for c_row in range(_n_columnas - 5):
            new_fila.append("")
        lista_calif_establ.loc[len(lista_calif_establ.index)] = new_fila

    # proceso de revisar por cada columna los distintos valores
    for i1 in range(len(nums_columnas_parm) - 1):
        val1 = nums_columnas_parm[i1]
        val2 = nums_columnas_parm[i1 + 1]
        
        # si la columna tiene maximo 3 subcolumnas
        interrango = 0
        if val2 - val1 >= 4:
            interrango = 3
        elif val2 - val1 == 3:
            interrango = 3
        elif val2 - val1 >= 2:
            interrango = 2
        
        for k2_id in lista_calif_establ['ID_Control'].tolist():
            namae_parametro = datos_subitems.loc[datos_subitems['Valores'] == get_key(parametros_pestana, val1), 'Nombre'].tolist()[0]  # nombre del parametro en el nuevo formato, de acuerdo a val1
            namae2 = ""
            namae3 = ""
            for r_param in columnas_totales:
                if namae_parametro in r_param and not "_VALORES" in r_param:
                    namae2 = r_param
                    namae3 = r_param + "_VALORES"
            valor_parametro = ""
            namae_param = ""
            for enum in range(val1, val1 + interrango):
                dato_calif = df.loc[df['ID_Control'] == k2_id, str(loc_columnas[enum])].tolist()[0]
                if dato_calif != "" and dato_calif != 0:
                    valor_parametro = valor_parametro + str(dato_calif) + ("," if valor_parametro == "" else "")
                    namae_param = namae_param + loc_calificaciones[enum] + ("," if namae_param == "" else "")
                    lista_calif_establ.loc[lista_calif_establ['ID_Control'] == k2_id, namae3] = valor_parametro
                    lista_calif_establ.loc[lista_calif_establ['ID_Control'] == k2_id, namae2] = namae_param
            lista_calif_establ.loc[lista_calif_establ['ID_Control'] == k2_id, 'Calificacion_Total'] = df.loc[df['ID_Control'] == k2_id, 'Cumplimiento'].tolist()[0]

        #print(get_key(parametros_pestana, val1))
        
    #todos_parametros.append([nombre_pestana, parametros_pestana])

    database = pd.concat([database, lista_calif_establ], axis= 0) 

database.to_excel(resultado1, index=False)

TIENDAS-EXPENDIOS
['Motivo', 'ID_Control', 'Fecha_visita', 'Fecha_ingreso', 'Razon_social', 'N_Documento', 'Nombre_comercial', 'N_Inscripcion', 'Mat_Mercantil', 'Direccion', 'Departamento', 'Municipio', 'Zona', 'Barrio', 'Telefono', 'Nombre_propietario', 'ID_propietario', 'Nombre_representante', 'Unnamed: 18', 'ID_propietario', 'Direccion_notif', 'EDIFICACIONES', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'EQUIPOS', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'PERSONAL', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'SANEAMIENTO', 'Unnamed: 58', 'Unnamed: 59', 'Unnamed: 60', 'Unnamed: 61', 'Unnamed: 62', 'Unnamed: 63', 'Unnamed: 6

ValueError: cannot set a row with mismatched columns

In [ ]:
"""prueba = pd.ExcelFile(datos2)
nombres = prueba.sheet_names
for i in nombres:
    print(i)"""

In [ ]:
"""datos_cols = pd.read_excel(columns, index_col= False)
writer = pd.ExcelWriter(resultado1, engine= 'xlsxwriter')
for nombre_ventana in nombres:
    print(nombre_ventana)
    df = pd.read_excel(datos2, index_col= False, sheet_name= nombre_ventana)
    df.drop(df.index[[0, 1]], inplace= True)
    print(df.shape)
    for i in range(98):
        nueva_col = datos_cols.loc[datos_cols['ID'] == i, 'Nuevo'].tolist()[0]
        df.rename(columns= {df.columns[i]: nueva_col})
    df.to_excel(writer, sheet_name= nombre_ventana, index= False)
writer.save()
writer.close()
    """

In [ ]:
datos_cols = pd.read_excel(columns, index_col= False)
total_datos = []
for nombre_ventana in nombres:
    print(nombre_ventana)
    df = pd.read_excel(datos2, index_col= False, sheet_name= nombre_ventana)
    df.drop(df.index[[1]], inplace= True)
    df.fillna(value="", inplace= True)
    todos_valores = df.loc[0, :].values.flatten().tolist()
    todos_valores = list(dict.fromkeys(todos_valores))
    for dato in todos_valores:
        if not dato in total_datos:
            total_datos.append(dato)

In [ ]:
for i2 in total_datos:
    print(i2)